# HDB Resale Flat Prices ETL Dataflow

This notebook implements a parameterized, repeatable data pipeline for the HDB engineering home test. It creates the raw master, profiling reports, cleaned and rejected datasets, transformed dataset, and final hashed dataset.

Run the notebook from top to bottom. Each Python cell is preceded by a short explanation of what it does and why it is required.

## Module 1: Parameterization

This module contains the values an operator is expected to change. Source filenames can contain any number of CSV files. The process dates are inclusive and must use `YYYY-MM` format. Output files are overwritten on each run so that rerunning the same batch is repeatable.

### Import the required Python libraries

In [1]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # Fallback for running the exported Python code outside Jupyter.
    def display(value):
        print(value)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

### Set the batch parameters

Change the values in this cell before running the notebook. `INPUT_FOLDER` and `OUTPUT_FOLDER` may also be changed if the files are stored in different directories.

In [2]:
# Source extract filenames: add or remove entries as required.
source_file_names = [
    "Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv",
    "Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv",
    "Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv",
    "Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv",
    "Resale flat prices based on registration date from Jan-2017 onwards.csv",
]

# Output filenames.
master_set_file_name = "resale_flat_prices_master_file.csv"
reject_set_file_name = "resale_flat_prices_reject_file.csv"
cleaned_data_file_name = "resale_flat_prices_cleaned_file.csv"
transformed_data_file_name = "resale_flat_prices_transformed_file.csv"
hashed_data_file_name = "resale_flat_prices_final_file.csv"

# Inclusive processing period in YYYY-MM format.
process_from_date = "2012-01"
process_to_date = "2016-12"

# Change these folders to suit the runtime environment.
INPUT_FOLDER = Path("/content")
OUTPUT_FOLDER = Path("/content")
PROFILE_FOLDER = OUTPUT_FOLDER / "data_profile_reports"

source_file_paths = [INPUT_FOLDER / name for name in source_file_names]
master_file_path = OUTPUT_FOLDER / master_set_file_name
reject_file_path = OUTPUT_FOLDER / reject_set_file_name
cleaned_file_path = OUTPUT_FOLDER / cleaned_data_file_name
transformed_file_path = OUTPUT_FOLDER / transformed_data_file_name
hashed_file_path = OUTPUT_FOLDER / hashed_data_file_name

process_start_month = pd.Period(process_from_date, freq="M")
process_end_month = pd.Period(process_to_date, freq="M")

if process_start_month > process_end_month:
    raise ValueError("process_from_date must be earlier than or equal to process_to_date")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
PROFILE_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"Source files configured: {len(source_file_paths)}")
print(f"Processing period: {process_start_month} to {process_end_month}")
print(f"Output folder: {OUTPUT_FOLDER}")

Source files configured: 5
Processing period: 2012-01 to 2016-12
Output folder: /content


## Module 2: Creation of the raw master dataset

The source extracts are combined by column name rather than column position. The union of all discovered columns is retained. `remaining_lease` is added as null only when a source file does not contain it; existing source values are not changed in this raw layer.

### Read and combine all configured source files

The expected HDB columns are placed first in a consistent order. Any additional columns discovered in a future extract are retained after them, satisfying the requirement to keep all attributes found in the files.

In [3]:
EXPECTED_COLUMNS = [
    "month",
    "town",
    "flat_type",
    "block",
    "street_name",
    "storey_range",
    "floor_area_sqm",
    "flat_model",
    "lease_commence_date",
    "remaining_lease",
    "resale_price",
]

source_frames = []
discovered_columns = []

for file_path in source_file_paths:
    if not file_path.exists():
        raise FileNotFoundError(f"Source file not found: {file_path}")

    source_df = pd.read_csv(file_path, low_memory=False)
    source_df.columns = source_df.columns.str.strip()

    for column_name in source_df.columns:
        if column_name not in discovered_columns:
            discovered_columns.append(column_name)

    source_frames.append(source_df)
    print(f"Loaded {file_path.name}: {len(source_df):,} rows, {len(source_df.columns)} columns")

if not source_frames:
    raise ValueError("No source files were configured")

extra_columns = [c for c in discovered_columns if c not in EXPECTED_COLUMNS]
MASTER_COLUMNS = EXPECTED_COLUMNS + extra_columns

aligned_frames = []
for source_df in source_frames:
    aligned_df = source_df.copy()
    for column_name in MASTER_COLUMNS:
        if column_name not in aligned_df.columns:
            aligned_df[column_name] = pd.NA
    aligned_frames.append(aligned_df[MASTER_COLUMNS])

combined_source_df = pd.concat(aligned_frames, ignore_index=True)

print(f"\nCombined source rows: {len(combined_source_df):,}")
print(f"Master schema columns: {len(MASTER_COLUMNS)}")
display(combined_source_df.head())

Loaded Resale Flat Prices (Based on Approval Date), 1990 - 1999.csv: 287,196 rows, 10 columns
Loaded Resale Flat Prices (Based on Approval Date), 2000 - Feb 2012.csv: 369,651 rows, 10 columns
Loaded Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv: 37,153 rows, 11 columns
Loaded Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv: 52,203 rows, 10 columns
Loaded Resale flat prices based on registration date from Jan-2017 onwards.csv: 235,247 rows, 11 columns

Combined source rows: 981,450
Master schema columns: 11


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,NaN,9000.0
1,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,04 TO 06,31.0,IMPROVED,1977,NaN,6000.0
2,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,10 TO 12,31.0,IMPROVED,1977,NaN,8000.0
3,1990-01,ANG MO KIO,1 ROOM,309,ANG MO KIO AVE 1,07 TO 09,31.0,IMPROVED,1977,NaN,6000.0
4,1990-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,04 TO 06,73.0,NEW GENERATION,1976,NaN,47200.0


### Apply the processing period, sort, and write the raw master

Only rows whose `month` falls within the inclusive processing period are written to the master file.

In [4]:
source_month_text = combined_source_df["month"].astype("string").str.strip()
source_month_parsed = pd.to_datetime(source_month_text, format="%Y-%m", errors="coerce")
source_month_period = source_month_parsed.dt.to_period("M")

within_process_period = source_month_period.between(
    process_start_month,
    process_end_month,
)

# Retained in memory for Module 4; no reject file is created here.
date_scope_rejected_df = combined_source_df.loc[~within_process_period].copy()
date_scope_rejected_df["_date_scope_reason"] = np.where(
    source_month_parsed.loc[~within_process_period].isna(),
    "Invalid or missing month; expected YYYY-MM",
    f"Month is outside processing period {process_start_month} to {process_end_month}",
)

master_df = combined_source_df.loc[within_process_period, MASTER_COLUMNS].copy()
master_df["_month_sort"] = source_month_parsed.loc[within_process_period]
master_df.sort_values("_month_sort", kind="mergesort", inplace=True)
master_df.drop(columns="_month_sort", inplace=True)
master_df.reset_index(drop=True, inplace=True)

# Overwrite mode is intentional for a repeatable batch result.
master_df.to_csv(master_file_path, index=False, mode="w")

print(f"Raw master rows written: {len(master_df):,}")
print(f"Rows held for date-scope rejection: {len(date_scope_rejected_df):,}")
print(f"Raw master file: {master_file_path}")
display(master_df.head())

Raw master rows written: 92,544
Rows held for date-scope rejection: 888,906
Raw master file: /content/resale_flat_prices_master_file.csv


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,NaN,257800.0
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,NaN,263000.0
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,NaN,275000.0
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,NaN,260000.0
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,NaN,226000.0


## Module 3: Data profiling

### Create dataset, column, category, numeric, and duplicate profiles

The profile outputs are plain CSV files.

In [5]:
profile_df = pd.read_csv(master_file_path, low_memory=False)

# 1. Column completeness and cardinality.
column_profile_rows = []
for column_name in profile_df.columns:
    series = profile_df[column_name]
    blank_count = int(series.astype("string").str.strip().eq("").fillna(False).sum())
    null_count = int(series.isna().sum())
    missing_count = null_count + blank_count

    column_profile_rows.append({
        "column_name": column_name,
        "pandas_dtype": str(series.dtype),
        "row_count": len(profile_df),
        "null_count": null_count,
        "blank_count": blank_count,
        "missing_percentage": round((missing_count / max(len(profile_df), 1)) * 100, 2),
        "distinct_count": int(series.nunique(dropna=True)),
    })

column_profile_df = pd.DataFrame(column_profile_rows)

# 2. Frequency distribution for the fields named in the validation requirement.
PROFILE_CATEGORY_COLUMNS = ["town", "flat_type", "flat_model", "storey_range"]
category_profile_frames = []

for column_name in PROFILE_CATEGORY_COLUMNS:
    value_counts = (
        profile_df[column_name]
        .astype("string")
        .str.strip()
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="record_count")
    )
    value_counts.insert(0, "column_name", column_name)
    value_counts["percentage"] = (
        value_counts["record_count"] / max(len(profile_df), 1) * 100
    ).round(2)
    category_profile_frames.append(value_counts)

category_profile_df = pd.concat(category_profile_frames, ignore_index=True)

# 3. Standard numeric summary for the main measurable fields.
numeric_columns = ["floor_area_sqm", "lease_commence_date", "resale_price"]
numeric_work_df = profile_df[numeric_columns].apply(pd.to_numeric, errors="coerce")
numeric_profile_df = (
    numeric_work_df.describe(percentiles=[0.25, 0.5, 0.75])
    .transpose()
    .reset_index()
    .rename(columns={"index": "column_name"})
)

# 4. One combined uniqueness and duplicate-price profile.
profile_key_columns = [c for c in profile_df.columns if c != "resale_price"]
duplicate_record_mask = profile_df.duplicated(subset=profile_key_columns, keep=False)

if duplicate_record_mask.any():
    duplicate_price_profile_df = (
        profile_df.loc[duplicate_record_mask]
        .groupby(profile_key_columns, dropna=False)["resale_price"]
        .agg(
            record_count="size",
            distinct_price_count="nunique",
            minimum_resale_price="min",
            maximum_resale_price="max",
        )
        .reset_index()
        .sort_values(["record_count", "distinct_price_count"], ascending=False)
    )
else:
    duplicate_price_profile_df = pd.DataFrame(
        columns=profile_key_columns + [
            "record_count",
            "distinct_price_count",
            "minimum_resale_price",
            "maximum_resale_price",
        ]
    )

# 5. Short rules-and-observations index for the reviewer.
profile_month = pd.to_datetime(
    profile_df["month"].astype("string").str.strip(),
    format="%Y-%m",
    errors="coerce",
)

profile_overview_df = pd.DataFrame([
    {
        "profile_area": "Volume",
        "rule_or_measure": "Count master rows and columns",
        "observation": f"{len(profile_df):,} rows and {len(profile_df.columns)} columns",
        "validation_use": "Provides the control total for reconciliation",
    },
    {
        "profile_area": "Completeness",
        "rule_or_measure": "Count null and blank values by column",
        "observation": f"{int((column_profile_df['missing_percentage'] > 0).sum())} columns contain at least one missing value",
        "validation_use": "Supports mandatory-field validation",
    },
    {
        "profile_area": "Date",
        "rule_or_measure": "Check parseability and observed month range",
        "observation": (
            f"Invalid months: {int(profile_month.isna().sum())}; "
            f"range: {profile_month.min():%Y-%m} to {profile_month.max():%Y-%m}"
            if profile_month.notna().any()
            else f"Invalid months: {len(profile_month)}; no valid month observed"
        ),
        "validation_use": "Supports strict YYYY-MM and process-period validation",
    },
    {
        "profile_area": "Categories",
        "rule_or_measure": "Count values for town, flat type, flat model and storey range",
        "observation": "; ".join(
            f"{column_name}: {profile_df[column_name].nunique(dropna=True)} distinct"
            for column_name in PROFILE_CATEGORY_COLUMNS
        ),
        "validation_use": "Values occurring at least twice form the supported reference set",
    },
    {
        "profile_area": "Numeric range",
        "rule_or_measure": "Calculate count, mean, quartiles, minimum and maximum",
        "observation": "See data_profile_numeric_summary.csv",
        "validation_use": "Supports positive-value checks and price anomaly analysis",
    },
    {
        "profile_area": "Uniqueness and price duplicates",
        "rule_or_measure": "Group by all columns except resale_price",
        "observation": (
            f"{int(duplicate_record_mask.sum()):,} records occur in duplicated composite-key groups; "
            f"{len(duplicate_price_profile_df):,} duplicate groups"
        ),
        "validation_use": "Supports retaining the highest resale price per composite key",
    },
])

profile_overview_df.to_csv(PROFILE_FOLDER / "data_profile_rules_and_observations.csv", index=False)
column_profile_df.to_csv(PROFILE_FOLDER / "data_profile_column_summary.csv", index=False)
category_profile_df.to_csv(PROFILE_FOLDER / "data_profile_category_frequencies.csv", index=False)
numeric_profile_df.to_csv(PROFILE_FOLDER / "data_profile_numeric_summary.csv", index=False)
duplicate_price_profile_df.to_csv(PROFILE_FOLDER / "data_profile_duplicate_price_groups.csv", index=False)

print(f"Profiling reports written to: {PROFILE_FOLDER}")
display(profile_overview_df)
display(column_profile_df)

Profiling reports written to: /content/data_profile_reports


,profile_area,rule_or_measure,observation,validation_use
0,Volume,Count master rows and columns,"92,544 rows and 11 columns",Provides the control total for reconciliation
1,Completeness,Count null and blank values by column,1 columns contain at least one missing value,Supports mandatory-field validation
2,Date,Check parseability and observed month range,Invalid months: 0; range: 2012-01 to 2016-12,Supports strict YYYY-MM and process-period validation
3,Categories,"Count values for town, flat type, flat model and storey range",town: 26 distinct; flat_type: 7 distinct; flat_model: 20 distinct; storey_range: 25 distinct,Values occurring at least twice form the supported reference set
4,Numeric range,"Calculate count, mean, quartiles, minimum and maximum",See data_profile_numeric_summary.csv,Supports positive-value checks and price anomaly analysis
5,Uniqueness and price duplicates,Group by all columns except resale_price,"3,154 records occur in duplicated composite-key groups; 1,559 duplicate groups",Supports retaining the highest resale price per composite key


,column_name,pandas_dtype,row_count,null_count,blank_count,missing_percentage,distinct_count
0,month,object,92544,0,0,0.00,60
1,town,object,92544,0,0,0.00,26
2,flat_type,object,92544,0,0,0.00,7
3,block,object,92544,0,0,0.00,2139
4,street_name,object,92544,0,0,0.00,522
5,storey_range,object,92544,0,0,0.00,25
6,floor_area_sqm,float64,92544,0,0,0.00,168
7,flat_model,object,92544,0,0,0.00,20
8,lease_commence_date,int64,92544,0,0,0.00,48
9,remaining_lease,float64,92544,55391,0,59.85,50


## Module 4: Data validation and cleaning

The named HDB fields are validated using the formats, ranges, and frequency distributions observed during profiling. A category is treated as statistically supported when its normalized value appears at least twice in the master dataset. If a dataset is very small and no value meets that threshold, all observed nonblank values are used.

The module also applies a few standard hard checks required for safe transformation: block and street must be present, floor area and resale price must be positive, and lease commencement must be a valid year.

### Build the profile-based reference values and apply validation rules

All applicable failure messages are collected, so a rejected record can show more than one reason. Text categories are trimmed, repeated spaces are collapsed, and values are standardized to uppercase in the cleaned output.

In [6]:
validation_df = pd.read_csv(master_file_path, low_memory=False)
validation_df = validation_df[MASTER_COLUMNS].copy()

MIN_SUPPORTED_FREQUENCY = 2


def normalize_text(series):
    # Trim, collapse repeated whitespace, and standardize text for comparison.
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )


def statistically_supported_values(series, minimum_frequency=2):
    # Return observed nonblank values meeting the frequency threshold.
    normalized = normalize_text(series)
    valid_values = normalized[normalized.notna() & normalized.ne("")]
    counts = valid_values.value_counts()
    supported = set(counts[counts >= minimum_frequency].index)
    return supported if supported else set(counts.index)


allowed_towns = statistically_supported_values(validation_df["town"], MIN_SUPPORTED_FREQUENCY)
allowed_flat_types = statistically_supported_values(validation_df["flat_type"], MIN_SUPPORTED_FREQUENCY)
allowed_flat_models = statistically_supported_values(validation_df["flat_model"], MIN_SUPPORTED_FREQUENCY)
allowed_storey_ranges = statistically_supported_values(validation_df["storey_range"], MIN_SUPPORTED_FREQUENCY)

month_text = validation_df["month"].astype("string").str.strip()
month_parsed = pd.to_datetime(month_text, format="%Y-%m", errors="coerce")
month_period = month_parsed.dt.to_period("M")

town_normalized = normalize_text(validation_df["town"])
flat_type_normalized = normalize_text(validation_df["flat_type"])
flat_model_normalized = normalize_text(validation_df["flat_model"])
storey_normalized = normalize_text(validation_df["storey_range"])

storey_parts = storey_normalized.str.extract(r"^(\d{2}) TO (\d{2})$")
storey_lower = pd.to_numeric(storey_parts[0], errors="coerce")
storey_upper = pd.to_numeric(storey_parts[1], errors="coerce")

lease_year = pd.to_numeric(validation_df["lease_commence_date"], errors="coerce")
floor_area = pd.to_numeric(validation_df["floor_area_sqm"], errors="coerce")
resale_price = pd.to_numeric(validation_df["resale_price"], errors="coerce")
current_year = pd.Timestamp.today().year

failure_reasons = [[] for _ in range(len(validation_df))]


def add_failure(mask, reason):
    # Append a reason to each record for which mask is True.
    for row_position in np.flatnonzero(np.asarray(mask.fillna(True), dtype=bool)):
        failure_reasons[row_position].append(reason)


valid_date = (
    month_text.str.fullmatch(r"\d{4}-\d{2}", na=False)
    & month_parsed.notna()
    & month_period.between(process_start_month, process_end_month)
)
add_failure(~valid_date, "month must be a valid YYYY-MM value within the processing period")

add_failure(~town_normalized.isin(allowed_towns), "town is missing or not statistically supported by the master dataset")
add_failure(~flat_type_normalized.isin(allowed_flat_types), "flat_type is missing or not statistically supported by the master dataset")
add_failure(~flat_model_normalized.isin(allowed_flat_models), "flat_model is missing or not statistically supported by the master dataset")

valid_storey = (
    storey_normalized.isin(allowed_storey_ranges)
    & storey_lower.notna()
    & storey_upper.notna()
    & storey_lower.le(storey_upper)
)
add_failure(~valid_storey, "storey_range must be an observed and correctly ordered NN TO NN value")

valid_lease_year = (
    lease_year.notna()
    & lease_year.mod(1).eq(0)
    & lease_year.between(1960, current_year)
)
add_failure(~valid_lease_year, f"lease_commence_date must be a whole year between 1960 and {current_year}")

block_text = validation_df["block"].astype("string").str.strip()
street_text = validation_df["street_name"].astype("string").str.strip()
add_failure(block_text.isna() | block_text.eq(""), "block is missing")
add_failure(street_text.isna() | street_text.eq(""), "street_name is missing")
add_failure(floor_area.isna() | floor_area.le(0), "floor_area_sqm must be numeric and greater than zero")
add_failure(resale_price.isna() | resale_price.le(0), "resale_price must be numeric and greater than zero")

validation_failed_mask = pd.Series([bool(reasons) for reasons in failure_reasons], index=validation_df.index)

field_reject_df = validation_df.loc[validation_failed_mask].copy()
field_reject_df["rejection_stage"] = "Data validation"
field_reject_df["rejection_reason"] = [
    "; ".join(failure_reasons[position])
    for position in np.flatnonzero(validation_failed_mask.to_numpy())
]

clean_candidate_df = validation_df.loc[~validation_failed_mask].copy()

# Apply the validated/standardized values to the clean candidates.
clean_candidate_df["month"] = month_parsed.loc[~validation_failed_mask].dt.strftime("%Y-%m")
clean_candidate_df["town"] = town_normalized.loc[~validation_failed_mask]
clean_candidate_df["flat_type"] = flat_type_normalized.loc[~validation_failed_mask]
clean_candidate_df["flat_model"] = flat_model_normalized.loc[~validation_failed_mask]
clean_candidate_df["storey_range"] = storey_normalized.loc[~validation_failed_mask]
clean_candidate_df["block"] = block_text.loc[~validation_failed_mask]
clean_candidate_df["street_name"] = street_text.loc[~validation_failed_mask]
clean_candidate_df["floor_area_sqm"] = floor_area.loc[~validation_failed_mask]
clean_candidate_df["lease_commence_date"] = lease_year.loc[~validation_failed_mask].astype("int64")
clean_candidate_df["resale_price"] = resale_price.loc[~validation_failed_mask]

print(f"Rows passing field validation: {len(clean_candidate_df):,}")
print(f"Rows failing field validation: {len(field_reject_df):,}")
display(field_reject_df)

Rows passing field validation: 92,543
Rows failing field validation: 1


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,rejection_stage,rejection_reason
88846,2016-10,PASIR RIS,2 ROOM,142,PASIR RIS ST 11,01 TO 03,55.0,2-room,1994,76.0,240000.0,Data validation,flat_model is missing or not statistically supported by the master dataset


### Recompute remaining lease as of the run date

The source contains only a lease commencement year, so the calculation assumes that each 99-year lease began on 1 January of that year. Only complete months remaining as of today are counted; partial months are rounded down. The result is formatted as years and months for every clean candidate.

In [7]:
as_of_date = pd.Timestamp.today().normalize()


def remaining_lease_as_of_today(commence_year):
    lease_start = pd.Timestamp(year=int(commence_year), month=1, day=1)
    lease_end = lease_start + pd.DateOffset(years=99)

    complete_months = (
        (lease_end.year - as_of_date.year) * 12
        + (lease_end.month - as_of_date.month)
        - int(lease_end.day < as_of_date.day)
    )
    complete_months = max(0, complete_months)

    years, months = divmod(complete_months, 12)
    return f"{years} years {months:02d} months"


clean_candidate_df["remaining_lease"] = clean_candidate_df["lease_commence_date"].apply(
    remaining_lease_as_of_today
)

print(f"Remaining lease recalculated as of: {as_of_date.date()}")
display(clean_candidate_df[["lease_commence_date", "remaining_lease"]].head())

Remaining lease recalculated as of: 2026-07-18


,lease_commence_date,remaining_lease
0,1979,51 years 05 months
1,1978,50 years 05 months
2,1978,50 years 05 months
3,1986,58 years 05 months
4,1986,58 years 05 months


### Create the composite key and retain the highest resale price

The composite key is formed from every master column except `resale_price`, after the remaining lease has been recomputed. It is kept as a readable pipe-delimited column for auditability. Within each duplicated key, records are ordered by resale price descending: the highest price is retained and all lower or redundant copies are rejected.

In [8]:
COMPOSITE_KEY_COLUMNS = [c for c in MASTER_COLUMNS if c != "resale_price"]


def create_composite_key(dataframe):
    key_values = dataframe[COMPOSITE_KEY_COLUMNS].astype("string").fillna("<NULL>")
    return key_values.agg("||".join, axis=1)


clean_candidate_df["composite_key"] = create_composite_key(clean_candidate_df)
clean_candidate_df.sort_values(
    ["composite_key", "resale_price"],
    ascending=[True, False],
    kind="mergesort",
    inplace=True,
)

duplicate_key_reject_mask = clean_candidate_df.duplicated(subset="composite_key", keep="first")
group_max_price = clean_candidate_df.groupby("composite_key")["resale_price"].transform("max")

duplicate_key_reject_df = clean_candidate_df.loc[duplicate_key_reject_mask].copy()
duplicate_key_reject_df["rejection_stage"] = "Composite-key deduplication"
duplicate_key_reject_df["rejection_reason"] = np.where(
    duplicate_key_reject_df["resale_price"].lt(group_max_price.loc[duplicate_key_reject_mask]),
    "Duplicate composite key; lower resale_price discarded",
    "Duplicate composite key and resale_price; one copy retained",
)

cleaned_df = clean_candidate_df.loc[~duplicate_key_reject_mask].copy()
cleaned_df.sort_values("month", kind="mergesort", inplace=True)
cleaned_df.reset_index(drop=True, inplace=True)

print(f"Composite-key duplicate rejects: {len(duplicate_key_reject_df):,}")
print(f"Rows remaining after deduplication: {len(cleaned_df):,}")

Composite-key duplicate rejects: 1,600
Rows remaining after deduplication: 90,943


### Consolidate rejects and write the cleaned and failed datasets

Date-scope exclusions from Module 2, field-validation failures, and composite-key duplicate failures are combined into one failed dataset. Every failed record includes a rejection stage and reason. Both required outputs use overwrite mode.

In [9]:
# Prepare the rows excluded by the raw date filter.
date_reject_df = date_scope_rejected_df[MASTER_COLUMNS].copy()
date_reject_df["composite_key"] = create_composite_key(date_reject_df)
date_reject_df["rejection_stage"] = "Date scope"
date_reject_df["rejection_reason"] = date_scope_rejected_df["_date_scope_reason"].to_numpy()

# Add an auditable composite key to field-validation rejects using their source values.
field_reject_df["composite_key"] = create_composite_key(field_reject_df)

REJECT_COLUMNS = MASTER_COLUMNS + ["composite_key", "rejection_stage", "rejection_reason"]

reject_df = pd.concat(
    [
        date_reject_df[REJECT_COLUMNS],
        field_reject_df[REJECT_COLUMNS],
        duplicate_key_reject_df[REJECT_COLUMNS],
    ],
    ignore_index=True,
)

CLEANED_COLUMNS = MASTER_COLUMNS + ["composite_key"]
cleaned_df = cleaned_df[CLEANED_COLUMNS]

cleaned_df.to_csv(cleaned_file_path, index=False, mode="w")
reject_df.to_csv(reject_file_path, index=False, mode="w")

if len(combined_source_df) != len(cleaned_df) + len(reject_df):
    raise AssertionError("Row reconciliation failed after data validation")

print(f"Cleaned rows written: {len(cleaned_df):,} -> {cleaned_file_path}")
print(f"Rejected rows written: {len(reject_df):,} -> {reject_file_path}")
print("Row reconciliation passed: combined source = cleaned + rejected")
display(reject_df[["rejection_stage", "rejection_reason"]].head())

Cleaned rows written: 90,943 -> /content/resale_flat_prices_cleaned_file.csv
Rejected rows written: 890,507 -> /content/resale_flat_prices_reject_file.csv
Row reconciliation passed: combined source = cleaned + rejected


,rejection_stage,rejection_reason
0,Date scope,Month is outside processing period 2012-01 to 2016-12
1,Date scope,Month is outside processing period 2012-01 to 2016-12
2,Date scope,Month is outside processing period 2012-01 to 2016-12
3,Date scope,Month is outside processing period 2012-01 to 2016-12
4,Date scope,Month is outside processing period 2012-01 to 2016-12


### Report potentially anomalous resale prices

Anomalies are reviewed rather than automatically rejected. The heuristic calculates resale price per square metre and compares it within year, town, and flat type. A record is flagged only when its group contains at least 20 records and it falls beyond three interquartile ranges from the group quartiles. This conservative rule identifies extreme values while reducing false positives.

In [10]:
anomaly_work_df = cleaned_df.copy()
anomaly_work_df["transaction_year"] = pd.to_datetime(
    anomaly_work_df["month"], format="%Y-%m"
).dt.year
anomaly_work_df["price_per_sqm"] = (
    anomaly_work_df["resale_price"] / anomaly_work_df["floor_area_sqm"]
)

anomaly_group_columns = ["transaction_year", "town", "flat_type"]
anomaly_group_stats = (
    anomaly_work_df.groupby(anomaly_group_columns)["price_per_sqm"]
    .agg(
        group_record_count="size",
        first_quartile=lambda values: values.quantile(0.25),
        third_quartile=lambda values: values.quantile(0.75),
    )
    .reset_index()
)
anomaly_group_stats["interquartile_range"] = (
    anomaly_group_stats["third_quartile"] - anomaly_group_stats["first_quartile"]
)
anomaly_group_stats["lower_bound"] = (
    anomaly_group_stats["first_quartile"] - 3 * anomaly_group_stats["interquartile_range"]
)
anomaly_group_stats["upper_bound"] = (
    anomaly_group_stats["third_quartile"] + 3 * anomaly_group_stats["interquartile_range"]
)

anomaly_work_df = anomaly_work_df.merge(anomaly_group_stats, on=anomaly_group_columns, how="left")
anomaly_mask = (
    anomaly_work_df["group_record_count"].ge(20)
    & (
        anomaly_work_df["price_per_sqm"].lt(anomaly_work_df["lower_bound"])
        | anomaly_work_df["price_per_sqm"].gt(anomaly_work_df["upper_bound"])
    )
)

price_anomaly_df = anomaly_work_df.loc[anomaly_mask].copy()
price_anomaly_df["anomaly_reason"] = "Price per sqm is outside the group's 3 x IQR bounds"
price_anomaly_df.to_csv(PROFILE_FOLDER / "resale_price_anomaly_report.csv", index=False)

print(f"Potential price anomalies reported (not rejected): {len(price_anomaly_df):,}")
print(f"Anomaly report: {PROFILE_FOLDER / 'resale_price_anomaly_report.csv'}")

Potential price anomalies reported (not rejected): 402
Anomaly report: /content/data_profile_reports/resale_price_anomaly_report.csv


## Module 5: Data transformation and hashing

This module reads the cleaned file created by Module 4 and builds the required `resale_identifier`. It then resolves duplicate identifiers by retaining the highest resale price. Transformation failures and discarded identifier duplicates are added to the existing reject file with clear reasons.

### Derive the unhashed resale identifier

The identifier has nine characters: `S` + three block digits + two average-price digits + two month digits + the town initial. The group average is calculated by month, town, and flat type. The average is rounded down to a whole dollar before taking its first two digits; this makes the assumption explicit and repeatable.

In [11]:
transformation_input_df = pd.read_csv(cleaned_file_path, low_memory=False)

transformation_input_df["resale_price"] = pd.to_numeric(
    transformation_input_df["resale_price"], errors="coerce"
)

block_digits = (
    transformation_input_df["block"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
)
block_code = block_digits.str[:3].str.zfill(3)

average_resale_price = transformation_input_df.groupby(
    ["month", "town", "flat_type"], dropna=False
)["resale_price"].transform("mean")
average_price_text = np.floor(average_resale_price).astype("Int64").astype("string")
average_price_code = average_price_text.str[:2]

transformation_month = pd.to_datetime(
    transformation_input_df["month"].astype("string").str.strip(),
    format="%Y-%m",
    errors="coerce",
)
month_code = transformation_month.dt.strftime("%m")
town_code = transformation_input_df["town"].astype("string").str.strip().str[:1]

transformation_failure = (
    block_digits.isna()
    | block_digits.eq("")
    | average_price_code.isna()
    | average_price_code.str.len().lt(2).fillna(True)
    | month_code.isna()
    | town_code.isna()
    | town_code.eq("")
)

transformation_reject_df = transformation_input_df.loc[transformation_failure].copy()
transformation_reject_df["rejection_stage"] = "Resale identifier transformation"
transformation_reject_df["rejection_reason"] = "Unable to derive one or more resale identifier components"

transformed_candidate_df = transformation_input_df.loc[~transformation_failure].copy()
transformed_candidate_df["resale_identifier"] = (
    "S"
    + block_code.loc[~transformation_failure]
    + average_price_code.loc[~transformation_failure]
    + month_code.loc[~transformation_failure]
    + town_code.loc[~transformation_failure]
)

if not transformed_candidate_df["resale_identifier"].str.fullmatch(r"S\d{7}[A-Z]").all():
    raise AssertionError("One or more resale identifiers do not follow the required format")

display(transformed_candidate_df[["block", "month", "town", "resale_identifier"]].head())

,block,month,town,resale_identifier
0,170,2012-01,ANG MO KIO,S1702501A
1,174,2012-01,ANG MO KIO,S1742501A
2,314,2012-01,ANG MO KIO,S3142501A
3,314,2012-01,ANG MO KIO,S3142501A
4,406,2012-01,ANG MO KIO,S4062501A


### Resolve duplicate resale identifiers and write the transformed dataset

The generated identifier is intentionally compact, so more than one transaction can produce the same value. To meet the uniqueness requirement, the highest-priced record for each identifier is retained. Lower-priced or redundant copies are added to the failed dataset. The failed file is rewritten with the consolidated contents, keeping the run repeatable.

In [12]:
transformed_candidate_df.sort_values(
    ["resale_identifier", "resale_price"],
    ascending=[True, False],
    kind="mergesort",
    inplace=True,
)

identifier_duplicate_mask = transformed_candidate_df.duplicated(
    subset="resale_identifier", keep="first"
)
identifier_group_max_price = transformed_candidate_df.groupby("resale_identifier")["resale_price"].transform("max")

identifier_reject_df = transformed_candidate_df.loc[identifier_duplicate_mask].copy()
identifier_reject_df["rejection_stage"] = "Resale identifier deduplication"
identifier_reject_df["rejection_reason"] = np.where(
    identifier_reject_df["resale_price"].lt(
        identifier_group_max_price.loc[identifier_duplicate_mask]
    ),
    "Duplicate resale_identifier; lower resale_price discarded",
    "Duplicate resale_identifier and resale_price; one copy retained",
)

transformed_df = transformed_candidate_df.loc[~identifier_duplicate_mask].copy()
transformed_df.sort_values("month", kind="mergesort", inplace=True)
transformed_df.reset_index(drop=True, inplace=True)

TRANSFORMED_COLUMNS = CLEANED_COLUMNS + ["resale_identifier"]
transformed_df = transformed_df[TRANSFORMED_COLUMNS]
transformed_df.to_csv(transformed_file_path, index=False, mode="w")

# Add Module 5 rejects to the failed dataset and rewrite the consolidated file.
existing_reject_df = pd.read_csv(reject_file_path, low_memory=False)

module_5_reject_frames = []
for module_5_df in [transformation_reject_df, identifier_reject_df]:
    prepared_df = module_5_df.copy()
    for column_name in REJECT_COLUMNS:
        if column_name not in prepared_df.columns:
            prepared_df[column_name] = pd.NA
    module_5_reject_frames.append(prepared_df[REJECT_COLUMNS])

final_reject_df = pd.concat(
    [existing_reject_df[REJECT_COLUMNS]] + module_5_reject_frames,
    ignore_index=True,
)
final_reject_df.to_csv(reject_file_path, index=False, mode="w")

if not transformed_df["resale_identifier"].is_unique:
    raise AssertionError("resale_identifier must be unique before hashing")

if len(cleaned_df) != len(transformed_df) + len(transformation_reject_df) + len(identifier_reject_df):
    raise AssertionError("Row reconciliation failed during transformation")

print(f"Transformed rows written: {len(transformed_df):,} -> {transformed_file_path}")
print(f"Module 5 rejects added: {len(transformation_reject_df) + len(identifier_reject_df):,}")
print(f"Consolidated reject rows: {len(final_reject_df):,} -> {reject_file_path}")

Transformed rows written: 77,254 -> /content/resale_flat_prices_transformed_file.csv
Module 5 rejects added: 13,689
Consolidated reject rows: 904,196 -> /content/resale_flat_prices_reject_file.csv


### Hash the identifier with SHA-256 and write the final dataset

The transformed file is read back as required. Each unhashed identifier is replaced with its SHA-256 hexadecimal digest. SHA-256 is deterministic and one-way for this use case, produces a fixed 64-character value, and has negligible collision risk. Because the input identifiers are unique, the code also asserts that the hashed values remain unique.

In [13]:
final_df = pd.read_csv(transformed_file_path, low_memory=False)

final_df["resale_identifier"] = final_df["resale_identifier"].apply(
    lambda value: hashlib.sha256(str(value).encode("utf-8")).hexdigest()
)

if not final_df["resale_identifier"].str.fullmatch(r"[0-9a-f]{64}").all():
    raise AssertionError("Hashed identifiers must be 64-character SHA-256 hexadecimal values")

if not final_df["resale_identifier"].is_unique:
    raise AssertionError("A hash collision or duplicate hashed identifier was detected")

final_df.to_csv(hashed_file_path, index=False, mode="w")

print(f"Final hashed rows written: {len(final_df):,} -> {hashed_file_path}")
display(final_df[["resale_identifier"]].head())

Final hashed rows written: 77,254 -> /content/resale_flat_prices_final_file.csv


,resale_identifier
0,84dee811ddd4db86fec49e7c609d67ba26895667338e3fdba693054b0d130a26
1,f7e32f86711abc69d5dfcd89b0a88efab285b816dd7c9174cff1d9c4f148b9c7
2,2bb55bf05c08ae97e208613abc12a863abe40996e6bc6e4ee64bb9978d749598
3,d814d99e285537934a9bfe8d3313784b05211086022d940faa7386728eea0da3
4,7f7ff5053bb29980b04069224dadc293dd2e4eff9e3ed9b5cf96317d41fc2576


## Run summary

The final cell displays the control totals and output locations. These checks give the reviewer a quick confirmation that every source record is accounted for and that the transformed and hashed datasets have equal row counts.

In [14]:
run_summary_df = pd.DataFrame([
    {"dataset": "Combined source input", "row_count": len(combined_source_df), "file": "Multiple configured CSV files"},
    {"dataset": "Raw master (selected date period)", "row_count": len(master_df), "file": str(master_file_path)},
    {"dataset": "Cleaned after validation", "row_count": len(cleaned_df), "file": str(cleaned_file_path)},
    {"dataset": "Rejected after all stages", "row_count": len(final_reject_df), "file": str(reject_file_path)},
    {"dataset": "Transformed", "row_count": len(transformed_df), "file": str(transformed_file_path)},
    {"dataset": "Final hashed", "row_count": len(final_df), "file": str(hashed_file_path)},
])

if len(combined_source_df) != len(transformed_df) + len(final_reject_df):
    raise AssertionError("Final row reconciliation failed")

if len(transformed_df) != len(final_df):
    raise AssertionError("Transformed and hashed row counts do not match")

display(run_summary_df)
print("Final reconciliation passed: combined source = transformed + all rejected records")

,dataset,row_count,file
0,Combined source input,981450,Multiple configured CSV files
1,Raw master (selected date period),92544,/content/resale_flat_prices_master_file.csv
2,Cleaned after validation,90943,/content/resale_flat_prices_cleaned_file.csv
3,Rejected after all stages,904196,/content/resale_flat_prices_reject_file.csv
4,Transformed,77254,/content/resale_flat_prices_transformed_file.csv
5,Final hashed,77254,/content/resale_flat_prices_final_file.csv


Final reconciliation passed: combined source = transformed + all rejected records
